In [31]:
import tensorflow as tf
import numpy as np
import random
import os
from copy import deepcopy
from sklearn.metrics import roc_auc_score

In [32]:
base_model = tf.keras.models.load_model('best_model.keras')

In [33]:
for layer in base_model.layers[:-20]:
    layer.trainable = False

In [34]:
n_way = 2
x = base_model.layers[-2].output
output = tf.keras.layers.Dense(n_way, activation="softmax")(x)
meta_model = tf.keras.Model(inputs=base_model.input, outputs=output)

In [35]:
loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
meta_optimizer = tf.keras.optimizers.Adam(1e-4)

## Sample Episodes from the Dataset

In [36]:
def sample_episode(dataset, n_way, k_shot, q_query):
    chosen_classes = random.sample(list(dataset.keys()), n_way)
    support_x, support_y, query_x, query_y = [], [], [], []

    for new_label, cls in enumerate(chosen_classes):
        samples = random.sample(dataset[cls], k_shot + q_query)
        s = samples[:k_shot]
        q = samples[k_shot:]
        support_x += [x for x, _ in s]
        support_y += [new_label]*len(s)
        query_x  += [x for x, _ in q]
        query_y  += [new_label]*len(q)

    return support_x.numpy(), np.array(support_y), query_x.numpy(), np.array(query_y)


## Reptile InnerLoop Training

In [37]:
def inner_train(model, support_x, support_y, inner_steps=5, inner_lr=1e-3):
    inner_model = tf.keras.models.clone_model(model)
    inner_model.set_weights(model.get_weights())
    opt = tf.keras.optimizers.Adam(inner_lr)

    for _ in range(inner_steps):
        with tf.GradientTape() as tape:
            preds = inner_model(support_x, training=True)
            loss = loss_fn(support_y, preds)
        grads = tape.gradient(loss, inner_model.trainable_variables)
        opt.apply_gradients(zip(grads, inner_model.trainable_variables))
    return inner_model

## Reptile Meta-Update

In [38]:
def reptile_update(model, inner_model, meta_step_size=0.1):
    new_weights = []
    for w_meta, w_inner in zip(model.get_weights(), inner_model.get_weights()):
        new_w = w_meta + meta_step_size * (w_inner - w_meta)
        new_weights.append(new_w)
    model.set_weights(new_weights)

## Meta Training Loop

In [39]:
def meta_train(meta_model, dataset, meta_iters=1000, n_way=2, k_shot=5, q_query=10, inner_steps=5, inner_lr=1e-3, meta_step_size=0.1):
    for it in range(meta_iters):
        # Sample episode
        support_x, support_y, query_x, query_y = sample_episode(dataset, n_way, k_shot, q_query)

        # Convert to tensors
        support_x = tf.convert_to_tensor(support_x, dtype=tf.float32)
        query_x   = tf.convert_to_tensor(query_x, dtype=tf.float32)

        # Inner adaptation
        inner_model = inner_train(meta_model, support_x, support_y, inner_steps, inner_lr)

        # Meta update
        reptile_update(meta_model, inner_model, meta_step_size)

        # Evaluate on query set occasionally
        if (it+1) % 50 == 0:
            preds = meta_model(query_x, training=False).numpy()
            auc = roc_auc_score(query_y, preds[:,1])
            print(f"Iter {it+1}: Episodic Query AUC = {auc:.4f}")

    meta_model.save("meta_reptile.keras")
    print("Meta-trained model saved as meta_reptile.keras")

## Few-Shot Adaption + Evaluation

In [40]:
def adapt_and_eval(meta_model, support_x, support_y, query_x, query_y, adapt_steps=20, adapt_lr=1e-4):
    adapted_model = tf.keras.models.clone_model(meta_model)
    adapted_model.set_weights(meta_model.get_weights())
    opt = tf.keras.optimizers.Adam(adapt_lr)

    # Adapt on support set
    for _ in range(adapt_steps):
        with tf.GradientTape() as tape:
            preds = adapted_model(support_x, training=True)
            loss = loss_fn(support_y, preds)
        grads = tape.gradient(loss, adapted_model.trainable_variables)
        opt.apply_gradients(zip(grads, adapted_model.trainable_variables))

    # Evaluate on query
    preds = adapted_model(query_x, training=False).numpy()
    auc = roc_auc_score(query_y, preds[:,1])
    print("Adapted model AUC:", auc)
    return auc

In [41]:
train_dir = "data/pediatric/train"
test_dir = "data/pediatric/test"

In [42]:
from keras.utils import load_img, img_to_array

def build_meta_dataset(base_dir, target_size=(224,224)):
    dataset = {}
    class_names = sorted(os.listdir(base_dir))  # each subfolder = class
    for class_idx, cls in enumerate(class_names):
        cls_path = os.path.join(base_dir, cls)
        if not os.path.isdir(cls_path):
            continue
        dataset[class_idx] = []
        for fname in os.listdir(cls_path):
            img_path = os.path.join(cls_path, fname)
            try:
                img = load_img(img_path, target_size=target_size)
                arr = img_to_array(img) / 255.0   # rescale like your datagen
                dataset[class_idx].append((arr, class_idx))
            except:
                pass
    return dataset

In [43]:
train_dataset = build_meta_dataset(train_dir)
test_dataset  = build_meta_dataset(test_dir)

In [44]:
meta_train(meta_model, train_dataset, meta_iters=500, n_way=2, k_shot=5, q_query=10)

Iter 50: Episodic Query AUC = 1.0000
Iter 100: Episodic Query AUC = 0.0200
Iter 150: Episodic Query AUC = 0.0000
Iter 200: Episodic Query AUC = 0.0400
Iter 250: Episodic Query AUC = 0.8900
Iter 300: Episodic Query AUC = 0.3100
Iter 350: Episodic Query AUC = 0.7800
Iter 400: Episodic Query AUC = 0.8500
Iter 450: Episodic Query AUC = 1.0000
Iter 500: Episodic Query AUC = 0.3200
Meta-trained model saved as meta_reptile.keras


In [45]:
support_x, support_y, query_x, query_y = sample_episode(test_dataset, 2, 5, 15)
support_x = tf.convert_to_tensor(support_x, tf.float32)
query_x   = tf.convert_to_tensor(query_x, tf.float32)

adapt_and_eval(meta_model, support_x, support_y, query_x, query_y)

Adapted model AUC: 0.8577777777777778


0.8577777777777778